# 实验前诊断

目标：确认数据是否具备解释实验结果的条件。先诊断，后推断。

In [12]:
import pandas as pd

events = pd.DataFrame({
    'event_id': [1, 2, 3, 4, 5, 6, 6, 7, 8, 9, 10],
    'user_id': [101, 101, 102, 102, 103, 104, 104, 105, 106, 107, 108],
    'group': ['control', 'control', 'treatment', 'treatment', 'control', 'treatment', 'treatment', None, 'control', 'control', 'treatment'],
    'event': ['checkout_view', 'payment_success', 'checkout_view', 'order_submitted', 'checkout_view', 'checkout_view', 'checkout_view', 'checkout_view', 'checkout_view', 'order_submitted', 'payment_success'],
})
events

,event_id,user_id,group,event
0,1,101,control,checkout_view
1,2,101,control,payment_success
2,3,102,treatment,checkout_view
3,4,102,treatment,order_submitted
4,5,103,control,checkout_view
5,6,104,treatment,checkout_view
6,6,104,treatment,checkout_view
7,7,105,None,checkout_view
8,8,106,control,checkout_view
9,9,107,control,order_submitted


## 数据质量

先计算风险数量；此时不要删除重复或缺失记录。

In [ ]:
# TODO：构造 quality，一行包含 rows、unique_users、missing_group、duplicate_event_id。
quality = pd.DataFrame(
    {
        "rows": [len(events)],
        "unique_users": [events['user_id'].nunique()],  # number of unique
        "missing_group": [events['group'].isnull().sum()],
        "duplicate_event_id": [events['event_id'].duplicated().sum()],
    }
)
quality

,rows,unique_users,missing_group,duplicate_event_id
0,11,8,1,1


## 分组规模

只对 group 非缺失的去重用户统计组别人数与占比。

In [34]:
# TODO：构造 group_sizes，包含 group、users、share。
# 1：过滤缺失值
valid_events = events.dropna(subset=["group"])

# 2: 分组聚合统计去重人数
group_sizes = (
    valid_events.groupby("group", as_index=False)
    .agg(users=("user_id", "nunique"))
)
display(group_sizes)

# 3：计算占比
group_sizes["share"] = group_sizes["users"] / group_sizes["users"].sum()

group_sizes

,group,users
0,control,4
1,treatment,3


,group,users,share
0,control,4,0.571429
1,treatment,3,0.428571


### 1. `.groupby("group", as_index=False)`

* **`groupby("group")`**：把数据按照 `group` 里的值拆成两堆（一堆是 `control` 对照组，一堆是 `treatment` 实验组）。
* **`as_index=False`（新手最容易忽略的细节）**：
  * **默认情况**：pandas 分组后会把 `"group"` 变成表格的行索引（Index），后续取数据或处理会不方便。
  * **加上参数**：告诉 pandas *“请把 `group` 依然保留为普通的数据列”*，这样生成的结果直接就是标准的表格形式（有表头、有正常的数据列）。

---

### 2. `.agg(users=("user_id", "nunique"))`

* **`.agg(...)`**：全称是 **aggregate（聚合）**，意思是“把每堆数据压缩成一个统计数值”。
* **参数语法拆解**：
  * **`users=`**：新生成的列名。这里我们将新列命名为 `users`。
  * **`"user_id"`**：指定拿哪一列来进行计算。
  * **`"nunique"`**：要执行的计算操作。`nunique` 即 **number of unique**（去重后的唯一值数量）。
  * **语义**：相当于告诉 pandas —— *“请统计每组里 `user_id` 去重后到底有几个人，算出来的列名叫做 `users`”*。

> **💡 为什么要用 `nunique` 而不是直接统计行数？**
> 
> 因为一个用户在一次实验中可能触发多个事件（例如：用户 `101` 既有 `checkout_view` 又有 `payment_success`），在日志表里会占多行。在做样本规模检查（Sample Ratio Mismatch / 规模核验）时，我们关心的是**有多少个独立真实用户（Unique Users）**，而不是产生了多少条行为日志。


## 漏斗

以 checkout_view 的去重用户为起点，按组计算各步骤的去重用户数与相对于 checkout_view 的比例。

In [31]:
# TODO：构造 funnel。提示：先排除 group 缺失，再按 group 和 event 计算 user_id.nunique()。
# 1. 过滤 group 缺失的用户
valid = events.dropna(subset=["group"])

# 2. 统计各组各环节的去重人数
funnel = (
    valid.groupby(["group", "event"], as_index=False)
    .agg(users=("user_id", "nunique"))
)
print("1")
print(funnel)

# 3. 统计各组 checkout_view（起点）的基准人数
base = funnel[funnel["event"] == "checkout_view"][["group", "users"]].rename(
    columns={"users": "base_users"}
)
print("2")
display(base)

# 4. 把基准人数拼回来，两列直接相除！
funnel = funnel.merge(base, on="group", how="left")
funnel["rate"] = funnel["users"] / funnel["base_users"]
print("3")
display(funnel)

# 5. 把辅助列删掉就完成了
funnel = funnel.drop(columns=["base_users"])
print("4")
funnel


1
       group            event  users
0    control    checkout_view      3
1    control  order_submitted      1
2    control  payment_success      1
3  treatment    checkout_view      2
4  treatment  order_submitted      1
5  treatment  payment_success      1
2


,group,base_users
0,control,3
3,treatment,2


3


,group,event,users,base_users,rate
0,control,checkout_view,3,3,1.000000
1,control,order_submitted,1,3,0.333333
2,control,payment_success,1,3,0.333333
3,treatment,checkout_view,2,2,1.000000
4,treatment,order_submitted,1,2,0.500000
5,treatment,payment_success,1,2,0.500000


4


,group,event,users,rate
0,control,checkout_view,3,1.000000
1,control,order_submitted,1,0.333333
2,control,payment_success,1,0.333333
3,treatment,checkout_view,2,1.000000
4,treatment,order_submitted,1,0.500000
5,treatment,payment_success,1,0.500000


## 诊断结论

填写指标口径、质量风险、还需核查的事实，以及一句结论边界。

结账页 A/B 实验前诊断报告
1. 指标口径（Metric Definition）：
分母：首次进入结账页（checkout_view）的去重用户数。
分子：观察窗口内成功提交订单（order_submitted）及支付成功（payment_success）的去重用户数。
2. 数据质量风险（Quality Risks）：
存在重复上报：event_id=6 重复记录了 1 次（共 11 行事件，去重事件 10 条）。
存在组别缺失：user_id=105 缺少分组标签（group is None）。
3. 业务与链路核查项（Needs Checking）：
漏斗断层/埋点丢失：对照组 user_id=107、实验组 user_id=108 均在未触发 checkout_view 的情况下直接产生了转化行为，需核查是埋点漏报、前端直购逻辑还是异常流量。
缺失原因归因：核查 105 用户的分组缺失是分流系统偶发失败还是客户端上报问题，评估排除后对样本平衡的影响。
4. 组别规模与失衡风险（SRM）：
当前有效去重用户对照组 4 人（57.1%）、实验组 3 人（42.9%）。样本量极小，需在扩大样本后核验卡方检验确认是否符合设计分配比。
5. 结论边界（Boundary）：
当前数据存在前序埋点漏报和组别缺失，转化率指标分母失真。现阶段仅完成实验前数据诊断，严禁直接依据当前转化率下达实验正向/负向的决策结论。